In [33]:
import pandas as pd
import numpy as np

In [34]:
billing_raw = pd.read_excel(r'C:\Users\pirat\Dropbox\Consulting Inc\Advanced Dermatology'
    r'\1. Payor Data & Contracts\Michigan\Aetna Analysis\Power BI - Aetna MI - 2024 & YTD Oct 2025 DOS - as of 1.29.26.xlsx')

service_map = pd.read_excel(r'C:\Users\pirat\Dropbox\Consulting Inc\Josh Perkins Data\CMS Data\Service Category Mapping.xlsx')

In [35]:
billing_clean = billing_raw.copy()

In [36]:
# replace col header spaces with underscores
billing_clean.columns = billing_clean.columns.str.replace(' ', '_')
service_map.columns = service_map.columns.str.replace(' ', '_')

In [37]:
# if "Provider SubGrp 1" is anything other than "Physicians" or "Extenders", change to "Physicians"
billing_clean.loc[~billing_clean['Provider_SubGrp_1'].isin(['Physicians', 'Extenders']), 'Provider_SubGrp_1'] = 'Physicians'

In [38]:
# New col "CPT Code" which is "Service Item" but only the characters before " - "
billing_clean['CPT_Code_Full'] = billing_clean['Service_Item'].str.split(' - ').str[0]

billing_clean['Service Description'] = billing_clean['Service_Item'].str.split(' - ').str[1]

billing_clean['CPT_Code_Core'] = billing_clean['CPT_Code_Full'].str.split('-').str[0]

In [39]:
# join service map "Service Category" to billing_clean on "CPT_Code_Core" and "CPT Code"
billing_clean = billing_clean.merge(service_map[['Code_Core', 'Category_Name']], 
                                    left_on='CPT_Code_Core', right_on='Code_Core', how='left')

In [40]:
# if year is missing, delete row
billing_clean = billing_clean.dropna(subset=['Year'])

# change year from float to int

billing_clean['Year'] = billing_clean['Year'].astype(int)

In [41]:
# Create date from 'Year' and 'Date_-_Month' columns with day as 1
billing_clean['Date'] = pd.to_datetime(billing_clean['Year'].astype(str) + '-' + billing_clean['Date_-_Month'].astype(str) + '-01')

In [42]:
# new col "Avg Alwd" == "Act_Alwd" / "Count"
billing_clean['Avg_Alwd'] = billing_clean['Act_Alwd'] / billing_clean['Count']

In [43]:
billing_clean['Total_Paid'] = (
    pd.to_numeric(billing_clean['Act_TP_Paid*'], errors='coerce')
      .add(pd.to_numeric(billing_clean['Pat_Paid'], errors='coerce'), fill_value=0)
      .round(5)
)*-1

billing_clean['Avg_Paid'] = billing_clean['Total_Paid'] / billing_clean['Count']

In [44]:
# Payor Mapping to LOB

# First make a series based on the raw insurance name
lob = billing_clean["Payer_Sub_Group_2"].fillna("").str.lower()

# build a guess for every row
payor_guess = np.select(
    [
        lob.str.contains("medicare") | lob.str.contains("mcr"),
        lob.str.contains("medicaid") | lob.str.contains("mcd"),
        lob.str.contains("tricare") | lob.str.contains("exchange"),
    ],
    ["MCR", "MCD", "Echange"],
    default="Commercial",
)

# map the guesses back to the dataframe
billing_clean["Payor_LOB"] = payor_guess

In [45]:
overrides = {
    # "AETNA Commercial FL Loc 99": "TEST"
}

billing_clean["Payor_LOB"] = (
    billing_clean["Contract_Name"].map(overrides)
    .fillna(billing_clean["Payor_LOB"])
)
